# Unit 3 — AutoGen Multi-Model + LangGraph + Gradio
## AI-Based Defect Reporting System
### Agentic AI & Automation — Symbiosis International University

**Learning Objectives (CO3):**
- Build Multiple AI Agents with AutoGen
- Make agents discuss ideas using AutoGen's GroupChat
- Core Components of LangGraph
- Agentic workflow in LangGraph
- Visualize Agentic workflow
- Connect to Gradio UI


In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Environment loaded')

## Part A: AutoGen Multi-Model AI Agents

In [ ]:
# AutoGen multi-agent team discussion
from app.agents.autogen_team import run_autogen_team_discussion

defect = 'User authentication silently fails — no error shown, user stuck on login page'

print('🤖 AutoGen Team Discussion')
print('Agents: Bug Analyst | QA Engineer | Project Manager')
print('='*60)

transcript = run_autogen_team_discussion(defect, severity='High')
print(transcript)

## Part B: LangGraph — Core Components

In [ ]:
# LangGraph core concepts demonstration
print('LangGraph Core Components:')
print('1. StateGraph  — Directed graph with shared state')
print('2. Nodes       — Python functions that transform state')
print('3. Edges       — Connections between nodes')
print('4. Conditional Edges — Route based on state values')
print('5. Entry Point — Starting node of the graph')
print('6. END         — Terminal node')
print()

# Show our workflow's nodes
print('Our Defect Reporting Workflow Nodes:')
nodes = [
    ('triage', 'Validates if input is a software defect (guardrail)'),
    ('analyze', 'Deep analysis using Gemini + tools + memory'),
    ('predict_severity', 'ML-based severity classification'),
    ('generate_report', 'Creates structured Markdown + JSON report'),
    ('reject', 'Handles non-defect inputs gracefully'),
]
for name, desc in nodes:
    print(f'  📦 {name}: {desc}')

In [ ]:
# Build and visualize the LangGraph workflow
from app.workflows.langgraph_workflow import build_workflow

graph = build_workflow()
if graph:
    print('✅ LangGraph workflow compiled successfully!')
    try:
        # Try to display the graph visualization
        from IPython.display import Image, display
        graph_image = graph.get_graph().draw_mermaid_png()
        display(Image(graph_image))
    except Exception as e:
        print(f'Visualization not available: {e}')
        print()
        print('Workflow Graph (Mermaid notation):')
        print('triage --> [valid] --> analyze')
        print('triage --> [invalid] --> reject')
        print('analyze --> predict_severity')
        print('predict_severity --> generate_report')
        print('generate_report --> END')
        print('reject --> END')
else:
    print('⚠️ LangGraph not installed — using fallback pipeline')

## Part C: Run the LangGraph Workflow

In [ ]:
from app.workflows.langgraph_workflow import run_workflow

test_cases = [
    'The checkout button freezes when clicked on mobile Safari',
    'What is the capital of France?'  # Should be rejected
]

for defect in test_cases:
    print(f'\n{"="*60}')
    print(f'Input: {defect}')
    state = run_workflow(defect)
    print(f'Final Step: {state["current_step"]}')
    print(f'Valid Defect: {state["is_valid_defect"]}')
    if state['is_valid_defect']:
        print(f'Severity: {state["severity"]}')
        print(f'Component: {state["component"]}')
    print(f'Report preview: {state["report_markdown"][:200]}...')

## Part D: Gradio UI for Workflow Visualization

In [ ]:
# Quick Gradio demo inline in notebook
import gradio as gr
from app.workflows.langgraph_workflow import run_workflow

def analyze_with_workflow(defect_text):
    state = run_workflow(defect_text)
    return (
        f'Valid: {state["is_valid_defect"]} | Category: {state["triage_category"]}',
        f'{state["severity"]} (Step: {state["current_step"]})',
        state['report_markdown'][:500]
    )

demo = gr.Interface(
    fn=analyze_with_workflow,
    inputs=gr.Textbox(label='Defect Description', lines=3),
    outputs=[
        gr.Textbox(label='Triage Result'),
        gr.Textbox(label='Severity'),
        gr.Textbox(label='Report Preview', lines=10),
    ],
    title='LangGraph Defect Workflow Demo (Unit 3)',
    examples=[
        ['API returns 404 for valid user accounts'],
        ['Search is very slow on mobile devices'],
    ]
)

print('Launching mini Gradio demo...')
demo.launch(inline=True, share=False)